<a href="https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/yolo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Traffic Sign Detection & Road Safety Monitor
### Cloud Training, Metric Verification & ONNX Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/yolo_colab.ipynb)

This notebook provides a complete training pipeline for **YOLOv8 Traffic Sign Object Detection**.

### Target Classes
- `0: stop` (Stop signs)
- `1: speed` (Speed limit signs: 20, 30, 50, 60, 70, 80, 100, 120 km/h)
- `2: danger` (Warning signs: curves, road works, pedestrian crossing, signals, etc.)

### Evaluation Targets
| Metric | Target Requirement |
|---|---|
| **mAP@0.5** | $\ge 0.80$ |
| **Missed-sign rate** (stop, speed, danger) | $\le 5\%$ |
| **CPU inference latency** | $\le 150$ ms/frame |
| **Deployed model size** | $\le 50$ MB |

## Step 1: Check GPU Acceleration
Make sure your Colab runtime is set to GPU (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists("traffic-sign-recognition-road-safety"):
    !git clone https://github.com/shubhamjena28/traffic-sign-recognition-road-safety.git

%cd traffic-sign-recognition-road-safety

# Install Ultralytics YOLO and ONNX export dependencies
!pip install --quiet ultralytics onnx onnxruntime onnxslim opencv-python matplotlib pandas pyyaml

## Step 3: Configure YOLO Dataset
Generates a Colab-compatible dataset configuration pointing to local paths.

In [ ]:
from pathlib import Path
import yaml

base_data = Path("/content/traffic-sign-recognition-road-safety/data")
if not base_data.exists():
    base_data = Path.cwd() / "data"

# Check if dataset images exist, if not, create starter annotated set
train_imgs = list((base_data / "images" / "train").glob("*.png")) + list((base_data / "images" / "train").glob("*.jpg"))
if len(train_imgs) == 0:
    print("[INFO] Generating starter annotated traffic sign dataset...")
    !python data/create_sample_dataset.py

# Write Colab dataset configuration
config = {
    "path": str(base_data.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "stop",
        1: "speed",
        2: "danger"
    }
}

colab_yaml_path = base_data / "traffic_signs_colab.yaml"
with open(colab_yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"[SUCCESS] Colab configuration written to: {colab_yaml_path}")
with open(colab_yaml_path) as f:
    print(f.read())

*(Optional)* **Download Full GTSDB Detection Dataset**
Uncomment below to download and convert the official German Traffic Sign Detection Benchmark archive (~1.6 GB) for full training.

In [ ]:
# !python data/download_datasets.py --dataset gtsdb
# !python data/convert_gtsdb_to_yolo.py --gtsdb-dir data/raw/gtsdb --output-dir data

## Step 4: Train YOLOv8 Detector
We start from the official pretrained `yolov8n.pt` (nano) weights. It satisfies the strict `< 50 MB` size target and `< 150 ms` latency requirement.

In [ ]:
from ultralytics import YOLO

# Initialize model
model = YOLO("yolov8n.pt")

# Run training
results = model.train(
    data=str(colab_yaml_path),
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/detect",
    name="colab_yolo_model",
    exist_ok=True,
    plots=True
)

## Step 5: Evaluate Against All Target Metrics

In [ ]:
import time
import numpy as np
from pathlib import Path
from ultralytics import YOLO

weights_path = Path("runs/detect/colab_yolo_model/weights/best.pt")
val_model = YOLO(str(weights_path))

# 1. Run validation
metrics = val_model.val(data=str(colab_yaml_path), imgsz=640, device="cpu")
map50 = float(metrics.box.map50)
recall = float(metrics.box.mr)
missed_rate = (1.0 - recall) * 100.0

# 2. Check deployed model size on disk
size_mb = weights_path.stat().st_size / (1024 * 1024)

# 3. Benchmark CPU Latency (50 frames)
test_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
for _ in range(5):
    _ = val_model.predict(test_img, device="cpu", verbose=False)

times = []
for _ in range(30):
    t0 = time.perf_counter()
    _ = val_model.predict(test_img, device="cpu", verbose=False)
    times.append((time.perf_counter() - t0) * 1000.0)

cpu_latency = float(np.mean(times))

print("\n" + "="*65)
print("             YOLO TRAFFIC SIGN BENCHMARK RESULTS")
print("="*65)
print(f"Metric                  Target          Achieved        Status")
print("-"*65)
status_map = "PASS" if map50 >= 0.80 else "TRAIN_MORE_EPOCHS"
status_miss = "PASS" if missed_rate <= 5.0 else "TRAIN_MORE_EPOCHS"
status_lat = "PASS" if cpu_latency <= 150.0 else "FAIL"
status_sz = "PASS" if size_mb <= 50.0 else "FAIL"

print(f"mAP@0.5                 >= 0.80         {map50:.4f}          {status_map}")
print(f"Missed-sign rate        <= 5.0%         {missed_rate:.2f}%          {status_miss}")
print(f"CPU Latency             <= 150 ms       {cpu_latency:.2f} ms        {status_lat}")
print(f"Model Size              <= 50 MB        {size_mb:.2f} MB         {status_sz}")
print("="*65)

## Step 6: Export Model to ONNX
Export the model to ONNX format for deployment with Java and OpenCV DNN.

In [ ]:
onnx_export_file = val_model.export(format="onnx", imgsz=640, opset=14, simplify=True)
onnx_file_path = Path(onnx_export_file)
onnx_size_mb = onnx_file_path.stat().st_size / (1024 * 1024)
print(f"[SUCCESS] Exported ONNX: {onnx_file_path} ({onnx_size_mb:.2f} MB)")

## Step 7: Visualize Results & Predictions

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

run_dir = Path("runs/detect/colab_yolo_model")
for fname in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    img_path = run_dir / fname
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(Image.open(img_path))
        plt.title(fname)
        plt.axis("off")
        plt.show()

## Step 8: Download Trained Weights
Downloads `best.pt` and `best.onnx` to your computer for immediate use in your Java and Python application.

In [ ]:
from google.colab import files

print("Downloading best.pt ...")
files.download(str(weights_path))

if onnx_file_path.exists():
    print("Downloading best.onnx ...")
    files.download(str(onnx_file_path))

print("[DONE] Both weight files downloaded.")